In [ ]:
from pathlib import Path
import sys

RAIZ = Path.cwd().resolve()
if not (RAIZ / "src" / "carregar_dados.py").is_file():
    RAIZ = RAIZ.parent
if not (RAIZ / "src" / "carregar_dados.py").is_file():
    raise RuntimeError("Abra o notebook na raiz do repositório ou em notebooks/.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))


In [ ]:
#1.Carregamento dos dados
import pandas as pd
import matplotlib.pyplot as plt
from src.carregar_dados import carregar_dados

df = carregar_dados()
print(df.shape)
print(df[['latitude', 'longitude']].dtypes)

In [ ]:
#2.Quais colunas são numéricas de vdd
colunas_numericas = df.select_dtypes(include='number').columns.tolist()

print(len(colunas_numericas))
print(colunas_numericas)

In [ ]:
alvo = 'valor_avaliacao'
identificadores = ['numero', 'cod_logradouro']
constantes = ['ano']
medidas = [c for c in colunas_numericas if c not in [alvo] + identificadores + constantes]

print(medidas)
print(df[colunas_numericas].nunique().sort_values())
print(df['logradouro'].nunique(), df['cod_logradouro'].nunique())

Das 11 colunas numericas, só 7 são medidas reais, já que **valor_avaliação** é o alvo, **numero e cod_logradouro** são só identificadores e o **ano** é igual pra todo mundo.

In [ ]:
#3.Estatísticas descritivas
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

tabela = df[medidas].describe().T
tabela['zeros'] = (df[medidas] == 0).sum()
tabela['nulos'] = df[medidas].isna().sum()
tabela

In [ ]:
#1. que tipo de imóvel tem terreno zero?
print(df[df['area_terreno'] == 0]['tipo_imovel'].value_counts())

#2. terreno "efetivo" de cada unidade
terreno_efetivo = df['area_terreno'] * df['fracao_ideal']
print(terreno_efetivo.describe())

Na base crua (16.318 linhas), nenhuma das 7 medidas tem ano impossível. As anomalias são a: **area_terreno** igual a zero em 86 linhas (0,5%), das quais 78 são casas, o que indica dado ausente registrado como zero; **fracao_ideal** igual a zero em 1 linha; **sfh** igual a zero em 65% das linhas; e **latitude/longitude** ausentes em 36% (5.850 linhas, sempre juntas). Em **area_terreno, area_construida, fracao_ideal e sfh** a média fica muito acima da mediana (assimetria à direita). Em geral a area_terreno representa o lote inteiro e a  fracao_ideal a é a parte de cada unidade usada pra construção do imovel: o produto das duas tem mediana de 25 m², contra 1.890 m² do terreno bruto.

In [ ]:
#4.Histogramas
fig, eixos = plt.subplots(2, 4, figsize=(18, 8))
eixos = eixos.flatten()

for i, col in enumerate(medidas):
    df[col].dropna().hist(bins=50, ax=eixos[i])
    eixos[i].set_title(col)

eixos[-1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
#1.pico de 2024/2025 e a barra de 1939
print((df['ano_construcao'] >= 2024).mean())
print((df['ano_construcao'] == 1939).sum())

#2.quem tem fração ideal igual a 1?
print((df['fracao_ideal'] == 1).sum())
print(df.loc[df['fracao_ideal'] == 1, 'tipo_imovel'].value_counts().head())

In [ ]:
import numpy as np

assimetricas = ['area_terreno', 'area_construida', 'fracao_ideal', 'sfh']

fig, eixos = plt.subplots(1, 4, figsize=(18, 4))

for ax, col in zip(eixos, assimetricas):
    positivos = df.loc[df[col] > 0, col]
    np.log10(positivos).hist(bins=50, ax=ax)
    ax.set_title(f'log10({col}), só valores > 0')

plt.tight_layout()
plt.show()

**area_terreno, area_construida, fracao_ideal e sfh** têm forte assimetria na direita: quase todos os imóveis estão perto de zero e poucos valores enormes esticam o eixo. **ano_construcao** tem um pico em 2024 e 2025 (cerca de 30% da base). **fracao_ideal** tem dois grupos, um perto de zero (unidades de prédio) e um em 1,0 (imóvel que ocupa o lote inteiro). **latitude e longitude** mostram dois aglomerados geográficos distintos.

In [ ]:
#5.Correlação com o alvo
import seaborn as sns

colunas_corr = medidas + [alvo]

pearson = df[colunas_corr].corr(method='pearson')
spearman = df[colunas_corr].corr(method='spearman')

fig, eixos = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(pearson, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=eixos[0])
eixos[0].set_title('Pearson')
sns.heatmap(spearman, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=eixos[1])
eixos[1].set_title('Spearman')
plt.tight_layout()
plt.show()

ranking = pd.DataFrame({'pearson': pearson[alvo], 'spearman': spearman[alvo]}).drop(alvo)
ranking['abs_spearman'] = ranking['spearman'].abs()
ranking.sort_values('abs_spearman', ascending=False)

area_construida é a única variável fortemente ligada ao alvo (Pearson 0,92 e Spearman 0,80; a diferença indica influência dos imóveis extremos). fracao_ideal tem relação moderada e não linear (Spearman 0,34 contra Pearson 0,17). As demais ficam próximas de zero (|ρ| ≤ 0,14). Há redundância entre atributos: area_terreno e fracao_ideal têm Spearman de -0,85. Em escala log-log, area_construida e valor_avaliacao formam uma nuvem diagonal estreita (valor aproximadamente proporcional à área), e pontos muito abaixo dela, como o imóvel de R$ 19,52, são candidatos a erro de registro. Na escala log10, area_terreno, area_construida e sfh ficam próximas de um sino simétrico, então a transformação log é candidata a pré-processamento.

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(14, 5))

#1. área construída x valor, em escala log
ok = (df['area_construida'] > 0) & (df['valor_avaliacao'] > 0)
eixos[0].scatter(np.log10(df.loc[ok, 'area_construida']),
                 np.log10(df.loc[ok, 'valor_avaliacao']), s=3, alpha=0.3)
eixos[0].set_xlabel('log10(area_construida)')
eixos[0].set_ylabel('log10(valor_avaliacao)')

#2. "mapa": cada ponto é um imóvel, colorido pelo valor
geo = df.dropna(subset=['latitude', 'longitude'])
g = eixos[1].scatter(geo['longitude'], geo['latitude'],
                     c=np.log10(geo['valor_avaliacao']), s=3, cmap='viridis')
plt.colorbar(g, ax=eixos[1], label='log10(valor_avaliacao)')
eixos[1].set_xlabel('longitude')
eixos[1].set_ylabel('latitude')

plt.tight_layout()
plt.show()

In [ ]:
geo = df.dropna(subset=['latitude', 'longitude'])
geo = geo[(geo['area_construida'] > 0) & (geo['valor_avaliacao'] > 0)]
valor_m2 = geo['valor_avaliacao'] / geo['area_construida']

vmin, vmax = valor_m2.quantile([0.05, 0.95])

plt.figure(figsize=(7, 6))
g = plt.scatter(geo['longitude'], geo['latitude'], c=valor_m2,
                s=3, cmap='viridis', vmin=vmin, vmax=vmax)
plt.colorbar(g, label='valor por m² (R$)')
plt.xlabel('longitude')
plt.ylabel('latitude')
plt.show()


print(df.loc[df['latitude'] < -8.09, 'bairro'].value_counts().head(5))
print(df.loc[df['latitude'] > -8.07, 'bairro'].value_counts().head(5))

In [ ]:
base = df[(df['area_construida'] > 0) & (df['valor_avaliacao'] > 0)].copy()
base['valor_m2'] = base['valor_avaliacao'] / base['area_construida']

por_bairro = base.groupby('bairro')['valor_m2'].agg(['median', 'count'])
por_bairro = por_bairro[por_bairro['count'] >= 30].sort_values('median')

print(por_bairro.head(5))
print(por_bairro.tail(5))

Pintando cada imóvel pelo valor por m² (usado só na EDA, nunca como atributo), da pra ver que o preço do m² varia bastante por região: há uma faixa mais cara no litoral sul e valores mais baixos no interior. Por bairro (mediana, mínimo de 30 imóveis), os mais caros são São José (R$ 7.497/m²), Poço, Pina, Monteiro e Casa Forte, e os mais baratos são Santo Antônio (R$ 1.517/m²), Areias, Ibura, Afogados e Recife. Apesar de a correlação linear de latitude e longitude com o alvo ser próxima de zero, a localização influencia o preço por região, o que favorece o kNN, que decide por proximidade.

In [ ]:
#6.Escalas e por que o kNN exige normalização
escalas = df[medidas].agg(['min', 'max']).T
escalas['amplitude'] = escalas['max'] - escalas['min']
escalas['desvio_padrao'] = df[medidas].std()
escalas.sort_values('amplitude', ascending=False)

In [ ]:
def participacao(dados):
    a = dados.sample(2000, random_state=42).values
    dif2 = (a[:1000] - a[1000:]) ** 2
    medias = dif2.mean(axis=0)
    return pd.Series(medias / medias.sum(), index=dados.columns)

sem_nulos = df[medidas].dropna()
padronizado = (sem_nulos - sem_nulos.mean()) / sem_nulos.std()

comparacao = pd.DataFrame({'sem escala': participacao(sem_nulos),
                           'padronizado': participacao(padronizado)})
(comparacao * 100).round(2)

As variáveis numéricas têm escalas muito diferentes: a amplitude vai de 4.000.000 (sfh) e 299.081 (area_terreno) até 0,15 (latitude) e 0,12 (longitude). Como o kNN usa distância euclidiana, sem normalização duas variáveis decidem tudo: em pares sorteados, o sfh respondeu por 95,5% da distância e o area_terreno por 4,5%, deixando as outras cinco com cerca de 0%, inclusive area_construida (a mais correlacionada com o alvo, Spearman 0,80) e as coordenadas. Com Z-score a participação fica entre 11% e 16% por variável. Concluímos que o kNN exige escalonamento. Latitude e longitude são medidas contínuas legítimas e permanecem como atributos.

In [ ]:
#7.Achados e variáveis que ficam

| Variável | Decisão | Motivo |
|---|---|---|
| `area_construida` | Manter | Principal preditor (Pearson 0,92, Spearman 0,80) |
| `area_terreno` | Manter | 86 zeros (78 são casas), tratar como faltante. Redundante com `fracao_ideal` (Spearman -0,85) |
| `fracao_ideal` | Manter | Relação moderada e não linear (Spearman 0,34) |
| `ano_construcao` | Manter | Medida real, sem nulos. Pico em 2024/25 (31%) e barra em 1939 (148 imóveis) |
| `latitude`, `longitude` | Manter | Medidas contínuas. Mapa mostra efeito regional. 36% nulos (tratamento: Pessoa 4) |
| `sfh` | Manter como candidata | Correlação ~0 e 65% de zeros. Significado a confirmar. Testar com e sem nos experimentos |
| `numero`, `cod_logradouro`, `ano` | Descartar | Identificadores e constante |
| `valor_avaliacao` | Alvo | Não é atributo |

Das 11 colunas numéricas, 7 são medidas reais; numero, cod_logradouro e ano são descartados (identificadores e constante). area_construida é o principal preditor do valor (Spearman 0,80), as demais têm correlação linear próxima de zero, mas a localização pesa por região (o m² varia muito entre bairros). area_terreno, area_construida, fracao_ideal e sfh são assimétricas à direita (o log as aproxima de um sino), e area_terreno e fracao_ideal são redundantes (Spearman -0,85). Como as escalas são muito diferentes, sem escalonamento o sfh responde por 95,5% da distância do kNN e area_construida por cerca de 0%, então a padronização é necessária e deve ser ajustada só no treino, dentro do Pipeline.